In [ ]:
# Ex2: Bias Audit of Pre-Trained Word Embeddings Using WEAT
!pip install gensim
!pip install gensim

import gensim
import numpy as np
from numpy.linalg import norm

# -------------------------------------------------
# Load Google News Word2Vec Model (2GB - takes time)
# -------------------------------------------------
import gensim.downloader as api

print("Loading pre-trained embedding...")
model = api.load("glove-wiki-gigaword-100")
print("Model loaded successfully")



# -------------------------------------------------
# Define Target Word Sets
# -------------------------------------------------
X = ["man", "male", "boy", "brother", "he", "him", "his"]
Y = ["woman", "female", "girl", "sister", "she", "her", "hers"]

# -------------------------------------------------
# Define Attribute Word Sets
# -------------------------------------------------
A = ["career", "business", "profession", "corporation", "salary", "office"]
B = ["family", "home", "children", "parents", "marriage", "house"]

# -------------------------------------------------
# Helper Functions
# -------------------------------------------------

def cosine_similarity(w1, w2):
    return np.dot(w1, w2) / (norm(w1) * norm(w2))

def association(w, A, B):
    return np.mean([cosine_similarity(model[w], model[a]) for a in A]) - \
           np.mean([cosine_similarity(model[w], model[b]) for b in B])

def weat_score(X, Y, A, B):
    return sum(association(x, A, B) for x in X) - \
           sum(association(y, A, B) for y in Y)

def effect_size(X, Y, A, B):
    assoc_X = [association(x, A, B) for x in X]
    assoc_Y = [association(y, A, B) for y in Y]
    mean_diff = np.mean(assoc_X) - np.mean(assoc_Y)
    std_dev = np.std(assoc_X + assoc_Y)
    return mean_diff / std_dev

# -------------------------------------------------
# Compute WEAT Score and Effect Size
# -------------------------------------------------
weat = weat_score(X, Y, A, B)
effect = effect_size(X, Y, A, B)

# -------------------------------------------------
# Display Results
# -------------------------------------------------
print("\nWEAT Score:", weat)
print("Effect Size:", effect)

if effect > 0:
    print("Career-related words are more associated with male terms")
elif effect < 0:
    print("Career-related words are more associated with female terms")
else:
    print("No significant gender bias detected")



Loading pre-trained embedding...
Model loaded successfully

WEAT Score: 0.39571917
Effect Size: 0.8411204
Career-related words are more associated with male terms
